# Deep Hedging — Brique 5 : remise en jambes PyTorch avec la CVaR

Objectif : revoir les essentiels de PyTorch (tenseurs, autograd, boucle d'entraînement) sur la **fonction objectif du projet**, la CVaR sous forme Rockafellar-Uryasev. On valide l'outil avant le couvreur neuronal.

Trois étapes :
1. autograd sur un exemple trivial (vérifier une dérivée),
2. minimiser la forme RU pour **retrouver la VaR et la CVaR**,
3. un couvreur à **un seul paramètre** `h` : le prototype du réseau.

Cibles numériques (validées à part en numpy) : étape 2, `w* ≈ 1.645`, `CVaR ≈ 2.06` ; étape 3, `h* ≈ 0.80`, `CVaR ≈ 0.62`.

In [ ]:
import torch
print("torch", torch.__version__)

## 1. Autograd : PyTorch calcule les dérivées pour toi

`requires_grad=True` marque un tenseur comme variable à différencier. Après `y.backward()`, le gradient est dans `x.grad`. Vérification sur `f(x)=x^3`, `f'(2)=12`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3
y.backward()                 # calcule dy/dx et le stocke dans x.grad
print("x.grad =", x.grad.item(), " (attendu 12)")

## 2. La CVaR (Rockafellar-Uryasev) comme fonction à minimiser

    CVaR_alpha(L) = min_w { w + (1/(1-alpha)) E[(L-w)+] }

`(L-w)+` est `relu(L-w)`. La fonction est convexe en `w`, donc la descente de gradient converge : à l'optimum, `w` est la VaR et la valeur est la CVaR. On l'écrit une fois, on la réutilisera pour le réseau.

In [ ]:
def cvar_ru(loss, w, alpha=0.95):
    """Forme RU de la CVaR : convexe en w, différentiable."""
    return w + torch.mean(torch.relu(loss - w)) / (1.0 - alpha)

torch.manual_seed(0)
L = torch.normal(0.0, 1.0, size=(200_000,))      # échantillon de pertes N(0,1)

w = torch.zeros(1, requires_grad=True)           # variable auxiliaire à optimiser
opt = torch.optim.Adam([w], lr=0.05)
for _ in range(2000):
    opt.zero_grad()                              # remet les gradients à zéro
    obj = cvar_ru(L, w)                           # objectif RU
    obj.backward()                                # gradients
    opt.step()                                    # un pas d'optimisation

print(f"w*   = {w.item():.4f}   (VaR empirique {torch.quantile(L,0.95).item():.4f})")
print(f"CVaR = {cvar_ru(L, w).item():.4f}   (théorie ~2.063)")

## 3. Un couvreur à un seul paramètre

Marché incomplet en une étape. On doit `L(h) = (beta - h) X + eps` : une exposition `beta` à l'actif `X`, plus un bruit `eps` **non couvrable**. On tient `h` unités de `X`. On minimise la CVaR conjointement sur `h` et `w`.

L'optimum théorique est `h* = beta` (on annule l'exposition couvrable), et la CVaR bute sur le plancher irréductible `sigma_eps * phi(z)/(1-alpha)`. Remplacer ce `h` scalaire par un **réseau** qui lit l'état du marché, c'est le Deep Hedging.

In [ ]:
beta, sigX, sigE, alpha = 0.8, 1.0, 0.3, 0.95
torch.manual_seed(0)
X   = torch.normal(0.0, sigX, size=(200_000,))
eps = torch.normal(0.0, sigE, size=(200_000,))

h = torch.zeros(1, requires_grad=True)           # le "couvreur" : ici un simple scalaire
w = torch.zeros(1, requires_grad=True)           # auxiliaire CVaR
opt = torch.optim.Adam([h, w], lr=0.01)
for _ in range(3000):
    opt.zero_grad()
    Lh = (beta - h) * X + eps                     # perte couverte
    obj = cvar_ru(Lh, w, alpha)
    obj.backward()
    opt.step()

Lh = (beta - h) * X + eps
print(f"h*   = {h.item():.4f}   (optimum beta = {beta})")
print(f"CVaR = {cvar_ru(Lh, w, alpha).item():.4f}   (plancher {sigE*0.10313/(1-alpha):.4f})")

## Le pont vers la phase 3

Ici `h` est un seul nombre. Dans le couvreur neuronal, on remplace `h` par la sortie d'un réseau `F_theta(état du marché en t_k)`, on déroule la couverture sur toute une trajectoire, et on minimise la même `cvar_ru` de la perte finale par rapport aux poids `theta`. Tout ce que tu viens d'écrire (la boucle `zero_grad / backward / step` et la perte CVaR) se transpose tel quel.